# <center> Superbowl 15 win curse analysis </center>

In [ ]:
#############################################
# Import packages / component programs      #
#############################################

import os, sys
import ipywidgets as widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output

# Repository name (used in path buildin):
repo_name='Scratchs_neighborhood_utils'
repo_path = os.path.join(os.getcwd().split(repo_name)[0],
                         repo_name
                        )

sys.path.insert(0,f"{repo_path}")

from _000_system_include.global_functions import global_parameters
from sports.nfl.anl01_15win_curse_significance.step01_pull_data import pull_data
from sports.nfl.anl01_15win_curse_significance.step02_shape_data_with_pandas import shape_data_pandas
from sports.nfl.anl01_15win_curse_significance.step02_shape_data_with_sql import shape_data_sql
from sports.nfl.anl01_15win_curse_significance.step03_visualize_explore import visualize_explore
from sports.nfl.anl01_15win_curse_significance.step04_statistics_significance import stats_analysis


#############################################
# Initialize Widgets                        #
#############################################

min_list_year=1999
max_list_year=2025

# Create dropdowns
start_year_dropdown = widgets.Dropdown(
        options=range(min_list_year,max_list_year+1),
        value=min_list_year,
        description='Start Year:',
    )
end_year_dropdown = widgets.Dropdown(
        options=range(min_list_year,max_list_year+1),
        value=max_list_year,
        description='End Year:',
    )

start_year=start_year_dropdown.value
end_year=end_year_dropdown.value
    
#Pull parameters for module
gparams = global_parameters()
    
# Run analysis steps
pull_data(gparams,min_list_year,max_list_year)
if use_pandas_for_step2:
    reg_wins_list, reg_wins_max = shape_data_pandas(gparams,start_year,end_year)
else:
    reg_wins_list, reg_wins_max = shape_data_sql(gparams,start_year,end_year)

pie_dropdown = widgets.Dropdown(
        options=reg_wins_list,
        value=reg_wins_max,
        description='Regular season wins:',
    )
stats_dropdown = widgets.Dropdown(
        options=reg_wins_list,
        value=reg_wins_max,
        description='Lowest regular season win count:',
    )

lowest_num_wins_pie=pie_dropdown.value
lowest_num_wins_stats=stats_dropdown.value

visualize_explore(gparams, lowest_num_wins_pie)
stats_string = stats_analysis(gparams, lowest_num_wins_stats)
clear_output()

# Create the output widget
out = widgets.Output(layout={'border': '1px solid black'})

# Create the output widget for pie chart
cat_out = widgets.Output(layout={'border': '1px solid black'})

# Create the output widget for pie chart
pct_bar_out = widgets.Output(layout={'border': '1px solid black'})

# Create the output widget for pie chart
pie_out = widgets.Output(layout={'border': '1px solid black'})

#############################################


#############################################
# Layouts                                   #
#############################################

# Place year drop downs in a horizaontal box
hbox_years = widgets.HBox(
    children=[start_year_dropdown, end_year_dropdown],
    layout=widgets.Layout(border='1px solid black')
)

# Component vertical box for pie chart
vbox_pie = widgets.VBox(
    children=[pie_dropdown, pie_out],
    layout=widgets.Layout(border='1px solid blue')
)

# Component vertical box for stats output
vbox_stats = widgets.VBox(
    children=[stats_dropdown, out],
    layout=widgets.Layout(border='1px solid blue')
)

# Vertical box for left pane (bar graphs)
vbox_left = widgets.VBox(
    children=[cat_out, pct_bar_out],
    layout=widgets.Layout(border='1px solid black')
)

# Vertical box for right pane (containing pie chart and stats output)
vbox_right = widgets.VBox(
    children=[vbox_pie, vbox_stats],
    layout=widgets.Layout(border='1px solid black')
)

# Component vertical box for pie chart
hbox_out = widgets.HBox(
    children=[vbox_left, vbox_right],
    layout=widgets.Layout(border='1px solid black')
)
#############################################



display(
    hbox_years, 
    hbox_out
)

with cat_out:
    fig_file_name="season_team_counts.png"
    fig_path=os.path.join(gparams['prog_path'],fig_file_name)
    clear_output()
    display(Image(filename=fig_path))
with pct_bar_out:
    fig_file_name="season_team_percents.png"
    fig_path=os.path.join(gparams['prog_path'],fig_file_name)
    clear_output()
    display(Image(filename=fig_path))

with pie_out:
    fig_file_name=f"pie_reg_wins_{lowest_num_wins_pie}_vs_15plus.png"
    fig_path=os.path.join(gparams['prog_path'],fig_file_name)
    clear_output()
    display(Image(filename=fig_path))
with out:
    clear_output()
    print(stats_string)

#########################################################


#############################################
# Define callback functions                 #
#############################################

def on_change_sy(change):
    start_year=start_year_dropdown.value
    end_year=end_year_dropdown.value
    if change['type'] == 'change' and change['name'] == 'value':
        start_year=change['new']
        if end_year != None and end_year < start_year:
            end_year = start_year
            end_year_dropdown.value = end_year
        
        #Pull parameters for module
        gparams = global_parameters()
        
        # Run analysis steps
        if use_pandas_for_step2:
            reg_wins_list, reg_wins_max = shape_data_pandas(gparams,start_year,end_year)
        else:
            reg_wins_list, reg_wins_max = shape_data_sql(gparams,start_year,end_year)
        
        pie_dropdown.options = reg_wins_list
        pie_dropdown.value = reg_wins_max  # Has to come after options assignment to override default
        stats_dropdown.options = reg_wins_list
        stats_dropdown.value = reg_wins_max  # Has to come after options assignment to override default
        lowest_num_wins_pie=pie_dropdown.value
        lowest_num_wins_stats=stats_dropdown.value

        visualize_explore(gparams, lowest_num_wins_pie)
        stats_string = stats_analysis(gparams, lowest_num_wins_stats)
        with cat_out:
            fig_file_name="season_team_counts.png"
            fig_path=os.path.join(gparams['prog_path'],fig_file_name)
            clear_output()
            display(Image(filename=fig_path))
        with pct_bar_out:
            fig_file_name="season_team_percents.png"
            fig_path=os.path.join(gparams['prog_path'],fig_file_name)
            clear_output()
            display(Image(filename=fig_path))
        with pie_out:
            fig_file_name=f"pie_reg_wins_{lowest_num_wins_pie}_vs_15plus.png"
            fig_path=os.path.join(gparams['prog_path'],fig_file_name)
            clear_output()
            display(Image(filename=fig_path))
        with out:
            clear_output()
            print(stats_string)

def on_change_ey(change):
    start_year=start_year_dropdown.value
    end_year=end_year_dropdown.value
    lowest_num_wins_pie=pie_dropdown.value
    lowest_num_wins_stats=stats_dropdown.value
    if change['type'] == 'change' and change['name'] == 'value':
        end_year=change['new']
        if start_year != None and end_year < start_year:
            start_year = end_year
            start_year_dropdown.value = start_year
        
        #Pull parameters for module
        gparams = global_parameters()
        
        # Run analysis steps
        if use_pandas_for_step2:
            reg_wins_list, reg_wins_max = shape_data_pandas(gparams,start_year,end_year)
        else:
            reg_wins_list, reg_wins_max = shape_data_sql(gparams,start_year,end_year)
        
        pie_dropdown.options = reg_wins_list
        pie_dropdown.value = reg_wins_max
        stats_dropdown.options = reg_wins_list
        stats_dropdown.value = reg_wins_max
        lowest_num_wins_pie=pie_dropdown.value
        lowest_num_wins_stats=stats_dropdown.value

        visualize_explore(gparams, lowest_num_wins_pie)
        stats_string = stats_analysis(gparams, lowest_num_wins_stats)
        with cat_out:
            fig_file_name="season_team_counts.png"
            fig_path=os.path.join(gparams['prog_path'],fig_file_name)
            clear_output()
            display(Image(filename=fig_path))
        with pct_bar_out:
            fig_file_name="season_team_percents.png"
            fig_path=os.path.join(gparams['prog_path'],fig_file_name)
            clear_output()
            display(Image(filename=fig_path))
        with pie_out:
            fig_file_name=f"pie_reg_wins_{lowest_num_wins_pie}_vs_15plus.png"
            fig_path=os.path.join(gparams['prog_path'],fig_file_name)
            clear_output()
            display(Image(filename=fig_path))
        with out:
            clear_output()
            print(stats_string)

def on_change_pie(change):
    if change['type'] == 'change' and change['name'] == 'value':
        lowest_num_wins_pie=change['new']
        
        #Pull parameters for module
        gparams = global_parameters()
        
        # Run analysis steps
        visualize_explore(gparams, lowest_num_wins_pie)
        with pie_out:
            fig_file_name=f"pie_reg_wins_{lowest_num_wins_pie}_vs_15plus.png"
            fig_path=os.path.join(gparams['prog_path'],fig_file_name)
            clear_output()
            display(Image(filename=fig_path))

def on_change_stats(change):
    if change['type'] == 'change' and change['name'] == 'value':
        lowest_num_wins_stats=change['new']
        
        #Pull parameters for module
        gparams = global_parameters()
        
        # Run analysis steps
        stats_string = stats_analysis(gparams, lowest_num_wins_stats)
        with out:
            clear_output()
            print(stats_string)

#############################################


########################################################
# Observe changes - link widgets to callback functions #
########################################################
start_year_dropdown.observe(on_change_sy)
end_year_dropdown.observe(on_change_ey)
pie_dropdown.observe(on_change_pie)
stats_dropdown.observe(on_change_stats)
########################################################
